In [76]:
# 그래프 관련
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
# matplotlib.rcParams['font.family'] = 'AppleGothic'
matplotlib.rcParams['axes.unicode_minus']=False

import pandas as pd
import numpy as np

# 머신러닝 관련
from sklearn import svm,metrics
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier  #평균절대값오차
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import mean_absolute_error
from sklearn.linear_model import Ridge
from sklearn.preprocessing import OneHotEncoder

In [77]:
## 다중회귀 - 특성이 여러 개일 때
df = pd.read_csv('data/MultipleLinearRegressionData.csv')
df.head(3)

,hour,absent,place,score
0,0.5,3,Home,10
1,1.2,4,Library,8
2,1.8,2,Cafe,14


In [78]:
data = df.iloc[:,:2]
data = np.array(data)
label = df['score']
place = df['place'].to_numpy().reshape(-1,1)
encoder = OneHotEncoder(sparse_output=False,drop='first')
result = encoder.fit_transform(place)
total_data = np.concatenate((data,result), axis=1)
total_data

array([[ 0.5,  3. ,  1. ,  0. ],
       [ 1.2,  4. ,  0. ,  1. ],
       [ 1.8,  2. ,  0. ,  0. ],
       [ 2.4,  0. ,  0. ,  0. ],
       [ 2.6,  2. ,  1. ,  0. ],
       [ 3.2,  0. ,  1. ,  0. ],
       [ 3.9,  0. ,  0. ,  1. ],
       [ 4.4,  0. ,  0. ,  1. ],
       [ 4.5,  5. ,  1. ,  0. ],
       [ 5. ,  1. ,  0. ,  0. ],
       [ 5.3,  2. ,  0. ,  0. ],
       [ 5.8,  0. ,  0. ,  0. ],
       [ 6. ,  3. ,  0. ,  1. ],
       [ 6.1,  1. ,  0. ,  0. ],
       [ 6.2,  1. ,  0. ,  1. ],
       [ 6.9,  4. ,  1. ,  0. ],
       [ 7.2,  2. ,  0. ,  0. ],
       [ 8.4,  1. ,  1. ,  0. ],
       [ 8.6,  1. ,  0. ,  1. ],
       [10. ,  0. ,  0. ,  1. ]])

In [79]:
train_data,test_data,train_label,test_label = train_test_split(total_data,label,random_state=20)

In [80]:
# 1차원
lr = LinearRegression()
lr.fit(train_data, train_label)
predict = lr.predict(test_data)
print('점수 예측: ', predict)

score = lr.score(train_data, train_label)
score2 = lr.score(test_data, test_label)
print(f'정확도:{score*100}%, {score2*100}%')

점수 예측:  [74.67906401 65.37800568 55.85968663 24.55625793 11.28991928]
정확도:96.78129791370543%, 98.8011581292527%


In [81]:
i_data = [[4.2, 1, 0, 0]] 

predict = lr.predict(i_data)
print('점수예측: ', predict)

점수예측:  [46.55862829]


In [82]:
poly = PolynomialFeatures(degree=2,include_bias=False)  #include_bias=False 학습의미없는 데이터 1를 생략
poly.fit(train_data)
train_poly = poly.transform(train_data)
test_poly = poly.transform(test_data)
train_poly[:2]

array([[ 6.2 ,  1.  ,  0.  ,  1.  , 38.44,  6.2 ,  0.  ,  6.2 ,  1.  ,
         0.  ,  1.  ,  0.  ,  0.  ,  1.  ],
       [ 8.4 ,  1.  ,  1.  ,  0.  , 70.56,  8.4 ,  8.4 ,  0.  ,  1.  ,
         1.  ,  0.  ,  1.  ,  0.  ,  0.  ]])

In [83]:
# 2차원
lr = LinearRegression()
lr.fit(train_poly, train_label)
predict = lr.predict(test_poly)
print('점수 예측: ', predict)

score = lr.score(train_poly, train_label)
score2 = lr.score(test_poly, test_label)
print(f'정확도:{score*100}%, {score2*100}%')

점수 예측:  [64.92000358 68.62600324 52.18994369 35.06552187 13.1344597 ]
정확도:99.9903813315809%, 89.4009753558838%


In [84]:
# StandartScaler
ss = StandardScaler()
ss.fit(train_poly)
train_scaled = ss.transform(train_poly)
test_scaled = ss.transform(test_poly)

In [85]:
alpha_list = [0.001,0.01,0.1,1,10,100]
train_score = []
test_score = []

for alpha in alpha_list:

    ridge = Ridge(alpha=alpha)
    ridge.fit(train_scaled,train_label)
    score = ridge.score(train_scaled, train_label)
    score2 = ridge.score(test_scaled, test_label)
    train_score.append(score)
    test_score.append(score2)
    
print(f'정확도:{score*100}%, {score2*100}%')


정확도:47.904073774716935%, 19.395676668203166%


In [86]:
# # 1. 새로운 데이터 준비 (2차원 리스트 형태)
# i_data = [[4.2, 1, 0, 0]]

# # 2. 훈련할 때 사용했던 poly 객체로 특성 뻥튀기 (반드시 transform만 사용)
# i_poly = poly.transform(i_data)

# # 3. 훈련할 때 사용했던 ss(StandardScaler) 객체로 스케일링 (반드시 transform만 사용)
# i_scaled = ss.transform(i_poly)

# # 4. 최종 예측
# predict = ridge.predict(i_scaled)
# print('예측 결과:', predict)

In [96]:
in_data = input('공부시간/결석횟수/공부장소(예:4.2,1,Cafe)')
in_data = in_data.split(',')
out_data = []
out_data.append(float(in_data[0]))
out_data.append(float(in_data[1]))


if in_data[2] == 'Home':
    out_data.append(1.0)
    out_data.append(0.0)
elif in_data[2] == 'Library':
    out_data.append(0.0)
    out_data.append(1.0)
elif in_data[2] == 'Cafe':
    out_data.append(0.0)
    out_data.append(0.0)
#print(out_data)


i_data = [out_data] # 2차원으로 만들기

# 훈련 모델에 맞게 변환 (poly와 ss 객체가 있다는 가정 하에)
i_poly = poly.transform(i_data)
i_scaled = ss.transform(i_poly)

# 최종 예측
predict = ridge.predict(i_scaled) # lr 대신 ridge 등 최종 모델 사용
print('예측결과:', predict)

예측결과: [49.27120458]
